# ELECTRA + ScalarMix + Domain Adversarial (text only)

**Self-contained:** model / GRL / ScalarMix code in this notebook.

**Data:** `full_text`, `education_level_judge`, `source_dataset` from `clean_dataset/` (+ optional **`coqa_train.csv`**).

**Colab:** Runtime → GPU → set paths in config → Run all. Saves to Drive when `SAVE_TO_DRIVE=True`.

- Phase 1: frozen encoder, λ=0
- Phase 2: gentle DANN (capped λ, scaled domain loss, partial freeze, early stop)


In [ ]:
!pip install transformers datasets scikit-learn torch pandas -q


In [ ]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

try:
    import google.colab  # noqa: F401

    RUNTIME = "colab"
    WORK_DIR = Path("/content")
except ImportError:
    RUNTIME = "kaggle" if Path("/kaggle").exists() else "local"
    WORK_DIR = Path("/kaggle/working") if RUNTIME == "kaggle" else Path.cwd().resolve()

print(f"RUNTIME={RUNTIME}  WORK_DIR={WORK_DIR}")


In [ ]:
if RUNTIME == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    print("Google Drive mounted.")


In [ ]:
# ========== ScalarMix (AllenNLP-style, inlined) ==========
from torch.nn import Parameter, ParameterList


class ConfigurationError(Exception):
    def __init__(self, message: str):
        super().__init__()
        self.message = message

    def __str__(self):
        return self.message


class ScalarMix(nn.Module):
    """gamma * sum softmax(w_k) * tensor_k across layers (do_layer_norm=False here)."""

    def __init__(
        self,
        mixture_size: int,
        do_layer_norm: bool = False,
        initial_scalar_parameters: Optional[List[float]] = None,
        trainable: bool = True,
    ) -> None:
        super().__init__()
        self.mixture_size = mixture_size
        self.do_layer_norm = do_layer_norm
        if initial_scalar_parameters is None:
            initial_scalar_parameters = [0.0] * mixture_size
        elif len(initial_scalar_parameters) != mixture_size:
            raise ConfigurationError(
                "Length of initial_scalar_parameters {} differs from mixture_size {}".format(
                    initial_scalar_parameters, mixture_size
                )
            )
        self.scalar_parameters = ParameterList(
            [
                Parameter(torch.FloatTensor([initial_scalar_parameters[i]]), requires_grad=trainable)
                for i in range(mixture_size)
            ]
        )
        self.gamma = Parameter(torch.FloatTensor([1.0]), requires_grad=trainable)

    @staticmethod
    def tiny_value_of_dtype(dtype: torch.dtype) -> float:
        if dtype in (torch.float, torch.double):
            return 1e-13
        if dtype == torch.half:
            return 1e-4
        raise TypeError("Only floating dtypes")

    def forward(
        self, tensors: List[torch.Tensor], mask: Optional[torch.BoolTensor] = None
    ) -> Tensor:
        if len(tensors) != self.mixture_size:
            raise ConfigurationError(
                "{} tensors were passed, but the module was initialized to mix {} tensors.".format(
                    len(tensors), self.mixture_size
                )
            )

        def _do_layer_norm(tensor, broadcast_mask, num_elements_not_masked):
            tensor_masked = tensor * broadcast_mask
            mean = torch.sum(tensor_masked) / num_elements_not_masked
            variance = (
                torch.sum(((tensor_masked - mean) * broadcast_mask) ** 2) / num_elements_not_masked
            )
            return (tensor - mean) / torch.sqrt(variance + self.tiny_value_of_dtype(variance.dtype))

        normed_weights = torch.nn.functional.softmax(
            torch.cat([parameter for parameter in self.scalar_parameters]), dim=0
        )
        normed_weights = torch.split(normed_weights, split_size_or_sections=1)

        if not self.do_layer_norm:
            pieces = []
            for weight, tensor in zip(normed_weights, tensors):
                pieces.append(weight * tensor)
            return self.gamma * sum(pieces)

        assert mask is not None
        broadcast_mask = mask.unsqueeze(-1)
        input_dim = tensors[0].size(-1)
        num_elements_not_masked = torch.sum(mask) * input_dim
        pieces = []
        for weight, tensor in zip(normed_weights, tensors):
            pieces.append(
                weight * _do_layer_norm(tensor, broadcast_mask, num_elements_not_masked)
            )
        return self.gamma * sum(pieces)

print("ScalarMix OK")


In [ ]:
# ========== Domain adversarial (GRL, heads, model, loss, eval) ==========


def grl_lambda_schedule(progress: float) -> float:
    progress = float(min(1.0, max(0.0, progress)))
    return 2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0


def phase2_grl_lambda(
    global_step: int,
    total_steps: int,
    *,
    lam_max: float,
    warmup_frac: float,
) -> float:
    total_steps = max(int(total_steps), 1)
    warmup_steps = int(round(float(warmup_frac) * total_steps))
    warmup_steps = min(warmup_steps, max(total_steps - 1, 0))
    if global_step < warmup_steps:
        return 0.0
    ramp_len = max(total_steps - warmup_steps, 1)
    p = (global_step - warmup_steps) / max(ramp_len - 1, 1)
    p = float(min(1.0, max(0.0, p)))
    return float(lam_max) * grl_lambda_schedule(p)


def set_encoder_freeze_bottom_layers(model: nn.Module, freeze_first_n: int) -> None:
    enc = model.encoder
    n_layers = len(enc.encoder.layer)
    freeze_first_n = max(0, min(int(freeze_first_n), n_layers))
    for p in enc.embeddings.parameters():
        p.requires_grad = False
    for i, layer in enumerate(enc.encoder.layer):
        req = i >= freeze_first_n
        for p in layer.parameters():
            p.requires_grad = req
    print(f"Encoder: bottom {freeze_first_n}/{n_layers} layers frozen")


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx: Any, x: Tensor, lambda_: float) -> Tensor:
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> Tuple[Tensor, None]:
        return -ctx.lambda_ * grad_output, None


def apply_gradient_reversal(x: Tensor, lambda_: float) -> Tensor:
    return GradientReversalFunction.apply(x, float(lambda_))


def build_domain2id(sources: Union[Sequence[str], Iterable[str]]) -> Tuple[Dict[str, int], Dict[int, str], int]:
    unique = sorted(set(str(s) for s in sources))
    domain2id: Dict[str, int] = {s: i for i, s in enumerate(unique)}
    id2domain: Dict[int, str] = {i: s for s, i in domain2id.items()}
    return domain2id, id2domain, len(unique)


class DomainClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_domains: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_domains),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class DifficultyClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 3, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class ElectraScalarMixDomainAdversarial(nn.Module):
    def __init__(
        self,
        model_name: str = "google/electra-large-discriminator",
        num_classes: int = 3,
        num_domains: int = 8,
        dropout: float = 0.2,
        scalar_mix_trainable: bool = True,
    ) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = int(self.encoder.config.hidden_size)
        n_layers = int(self.encoder.config.num_hidden_layers) + 1
        self.scalar_mix = ScalarMix(n_layers, trainable=scalar_mix_trainable)
        self.dropout = nn.Dropout(dropout)
        self.difficulty_head = DifficultyClassifierHead(hidden, num_classes, dropout)
        self.domain_head = DomainClassifierHead(hidden, num_domains, dropout)
        self.hidden_size = hidden

    def encode_pooled(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        mixed = self.scalar_mix(list(out.hidden_states))
        mixed = self.dropout(mixed)
        mask = attention_mask.unsqueeze(-1).float()
        summed = (mixed * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids: Tensor, attention_mask: Tensor, grl_lambda: float) -> Tuple[Tensor, Tensor]:
        pooled = self.encode_pooled(input_ids, attention_mask)
        difficulty_logits = self.difficulty_head(pooled)
        pooled_grl = apply_gradient_reversal(pooled, grl_lambda)
        domain_logits = self.domain_head(pooled_grl)
        return difficulty_logits, domain_logits


def combined_adversarial_loss(
    difficulty_logits: Tensor,
    domain_logits: Tensor,
    labels: Tensor,
    domain_labels: Tensor,
    *,
    label_smoothing: float = 0.1,
    domain_loss_weight: float = 1.0,
) -> Tuple[Tensor, Tensor, Tensor]:
    ce_task = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    ce_dom = nn.CrossEntropyLoss()
    difficulty_loss = ce_task(difficulty_logits, labels)
    domain_loss = ce_dom(domain_logits, domain_labels)
    total_loss = difficulty_loss + float(domain_loss_weight) * domain_loss
    return total_loss, difficulty_loss, domain_loss


@torch.no_grad()
def evaluate_difficulty_macro_f1_and_domain_accuracy(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
    *,
    grl_lambda: float = 0.0,
    label_key: str = "label",
    domain_key: str = "domain",
) -> Dict[str, float]:
    model.eval()
    all_y, all_py, all_d, all_pd = [], [], [], []
    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        y = batch[label_key].to(device)
        d = batch[domain_key].to(device)
        diff_logits, dom_logits = model(input_ids, attention_mask, grl_lambda=grl_lambda)
        all_y.extend(y.cpu().numpy().tolist())
        all_py.extend(torch.argmax(diff_logits, dim=1).cpu().numpy().tolist())
        all_d.extend(d.cpu().numpy().tolist())
        all_pd.extend(torch.argmax(dom_logits, dim=1).cpu().numpy().tolist())
    return {
        "macro_f1_difficulty": float(f1_score(all_y, all_py, average="macro")),
        "domain_accuracy": float(accuracy_score(all_d, all_pd)),
    }


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
MODEL_NAME = "google/electra-large-discriminator"
MAX_LEN = 512
BATCH_SIZE = 4
LABEL_SMOOTHING = 0.1

PHASE1_EPOCHS = 2
PHASE1_LR = 1e-3

PHASE2_EPOCHS = 2
PHASE2_ENCODER_LR = 5e-6
PHASE2_HEAD_LR = 1e-5
GRL_LAMBDA_MAX = 0.15
DOMAIN_LOSS_ALPHA = 0.1
PHASE2_LAMBDA_WARMUP_FRAC = 0.40
PHASE2_FREEZE_ENCODER_LAYERS = 8
PHASE2_EARLY_STOP_PATIENCE = 2

BALANCE_TRAIN = True
LABEL_COL = "education_level_judge"
SOURCE_COL = "source_dataset"
TEXT_COL = "full_text"

MANUAL_CLEAN_DIR = Path("/content/drive/MyDrive/beyond_flesch/clean_dataset")
COQA_CSV = Path("/content/drive/MyDrive/beyond_flesch/coqa_judge/clean_dataset/coqa_train.csv")
MERGE_COQA_INTO_TRAIN = True

SAVE_TO_DRIVE = True
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/beyond_flesch/electra_scalar_dann")

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [ ]:
# ── Resolve CLEAN_DIR ────────────────────────────────────────────────────
_candidates = []
if RUNTIME == "colab":
    _candidates.append(Path(MANUAL_CLEAN_DIR).expanduser().resolve())
_clone_clean = (
    WORK_DIR
    / "Beyond-Flesch"
    / "logistic-regression"
    / "llm_as_a_judge"
    / "llm_as_a_judge"
    / "outputs"
    / "clean_dataset"
)
_local_clean = (
    Path.cwd().resolve().parent
    / "llm_as_a_judge"
    / "llm_as_a_judge"
    / "outputs"
    / "clean_dataset"
)
_candidates.extend([_clone_clean, _local_clean, Path("/kaggle/input/clean-dataset")])

CLEAN_DIR = None
for d in _candidates:
    if (d / "train.csv").exists():
        CLEAN_DIR = d
        break
if CLEAN_DIR is None:
    raise FileNotFoundError(
        "Could not find train.csv. On Colab set MANUAL_CLEAN_DIR to your Drive clean_dataset folder."
    )
print("CLEAN_DIR:", CLEAN_DIR)


def load_split(name: str) -> pd.DataFrame:
    path = CLEAN_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    df = pd.read_csv(path)
    df = df.dropna(subset=[TEXT_COL, LABEL_COL, SOURCE_COL])
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    df = df[df[LABEL_COL].isin(label2id)]
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    df[SOURCE_COL] = df[SOURCE_COL].astype(str)
    return df


df_train = load_split("train")
df_val = load_split("val")
df_test = load_split("test")

n_before = len(df_train)
if MERGE_COQA_INTO_TRAIN:
    coqa_path = Path(COQA_CSV)
    if not coqa_path.exists():
        coqa_path = CLEAN_DIR / "coqa_train.csv"
    if coqa_path.exists():
        df_coqa = pd.read_csv(coqa_path)
        df_coqa = df_coqa.dropna(subset=[TEXT_COL, LABEL_COL, SOURCE_COL])
        df_coqa[LABEL_COL] = df_coqa[LABEL_COL].astype(str).str.strip().str.lower()
        df_coqa = df_coqa[df_coqa[LABEL_COL].isin(label2id)]
        df_coqa[TEXT_COL] = df_coqa[TEXT_COL].astype(str)
        df_coqa[SOURCE_COL] = df_coqa[SOURCE_COL].fillna("coqa").astype(str)
        df_train = pd.concat([df_train, df_coqa], ignore_index=True)
        print(f"Merged CoQA: +{len(df_coqa)} rows ({n_before} -> {len(df_train)}) from {coqa_path}")
    else:
        print(
            f"WARN: MERGE_COQA_INTO_TRAIN=True but not found: {COQA_CSV} or {CLEAN_DIR / 'coqa_train.csv'}"
        )

ood_splits = {}
for ood_name in ["ood_onestop", "ood_race-middle", "ood_race-high"]:
    p = CLEAN_DIR / f"{ood_name}.csv"
    if p.exists():
        ood_splits[ood_name] = load_split(ood_name)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print("Train judge labels:\n", df_train[LABEL_COL].value_counts())
print("Train sources:\n", df_train[SOURCE_COL].value_counts())


In [ ]:
_domain_sources = pd.concat([df_train[SOURCE_COL], df_val[SOURCE_COL]]).tolist()
domain2id, id2domain, num_domains = build_domain2id(_domain_sources)
print(f"num_domains={num_domains}")

if BALANCE_TRAIN:
    min_n = df_train[LABEL_COL].value_counts().min()
    print(f"Balancing train to {min_n} per class")
    parts = []
    for lab in label2id:
        sub = df_train[df_train[LABEL_COL] == lab]
        parts.append(sub.sample(n=min(min_n, len(sub)), random_state=42))
    df_train = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
    print(df_train[LABEL_COL].value_counts())


In [ ]:
class TextDomainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.texts = df[TEXT_COL].tolist()
        self.labels = [label2id[l] for l in df[LABEL_COL]]
        unknown_id = domain2id.get("__unknown__", 0)
        self.domains = [domain2id.get(s, unknown_id) for s in df[SOURCE_COL]]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "domain": torch.tensor(self.domains[idx], dtype=torch.long),
        }


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = TextDomainDataset(df_train, tokenizer, MAX_LEN)
val_ds = TextDomainDataset(df_val, tokenizer, MAX_LEN)
test_ds = TextDomainDataset(df_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2)

ood_loaders = {}
for name, odf in ood_splits.items():
    ood_loaders[name] = DataLoader(
        TextDomainDataset(odf, tokenizer, MAX_LEN), batch_size=BATCH_SIZE * 2
    )
print(f"Train batches: {len(train_loader)}")


In [ ]:
model = ElectraScalarMixDomainAdversarial(
    model_name=MODEL_NAME,
    num_classes=3,
    num_domains=num_domains,
    dropout=0.2,
).to(device)
print(f"Hidden size: {model.hidden_size}")


In [ ]:
def run_epoch(
    loader,
    optimizer,
    scheduler,
    global_step,
    total_steps,
    train_mode=True,
    grl_in_phase=False,
    lambda_fn=None,
    domain_loss_alpha=1.0,
):
    if train_mode:
        model.train()
    else:
        model.eval()

    total_loss = diff_sum = dom_sum = 0.0
    n_batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        domains = batch["domain"].to(device)

        if grl_in_phase and lambda_fn is not None:
            lam = lambda_fn(global_step)
        elif grl_in_phase:
            progress = global_step / max(total_steps - 1, 1)
            lam = grl_lambda_schedule(progress)
        else:
            lam = 0.0

        diff_logits, dom_logits = model(input_ids, attention_mask, grl_lambda=lam)
        loss, l_diff, l_dom = combined_adversarial_loss(
            diff_logits,
            dom_logits,
            labels,
            domains,
            label_smoothing=LABEL_SMOOTHING,
            domain_loss_weight=domain_loss_alpha,
        )

        if train_mode:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            global_step += 1

        total_loss += loss.item()
        diff_sum += l_diff.item()
        dom_sum += l_dom.item()
        n_batches += 1

    avg = {
        "loss": total_loss / max(n_batches, 1),
        "difficulty": diff_sum / max(n_batches, 1),
        "domain": dom_sum / max(n_batches, 1),
    }
    return avg, global_step


In [ ]:
print("Phase 1 — encoder frozen, λ=0")
for p in model.encoder.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=PHASE1_LR
)

total_steps_p1 = len(train_loader) * PHASE1_EPOCHS
global_step = 0

for epoch in range(PHASE1_EPOCHS):
    stats, global_step = run_epoch(
        train_loader, optimizer, None, global_step, total_steps_p1,
        train_mode=True, grl_in_phase=False,
    )
    print(f"  P1 epoch {epoch+1}: loss={stats['loss']:.4f} diff={stats['difficulty']:.4f} dom={stats['domain']:.4f}")
    m = evaluate_difficulty_macro_f1_and_domain_accuracy(model, val_loader, device)
    print(f"    val macro-F1={m['macro_f1_difficulty']:.4f} domain_acc={m['domain_accuracy']:.4f}")


In [ ]:
print("\nPhase 2 — gentle DANN")
for p in model.encoder.parameters():
    p.requires_grad = True
set_encoder_freeze_bottom_layers(model, PHASE2_FREEZE_ENCODER_LAYERS)

enc_params = [p for p in model.encoder.parameters() if p.requires_grad]
other_params = [
    p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")
]
optimizer = torch.optim.AdamW(
    [
        {"params": enc_params, "lr": PHASE2_ENCODER_LR, "weight_decay": 0.01},
        {"params": other_params, "lr": PHASE2_HEAD_LR, "weight_decay": 0.01},
    ]
)
total_steps_p2 = len(train_loader) * PHASE2_EPOCHS
global_step = 0
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps_p2 // 10,
    num_training_steps=total_steps_p2,
)

best_val_f1 = -1.0
best_state = None
stall = 0

for epoch in range(PHASE2_EPOCHS):
    def _p2_lambda(step: int) -> float:
        return phase2_grl_lambda(
            step,
            total_steps_p2,
            lam_max=GRL_LAMBDA_MAX,
            warmup_frac=PHASE2_LAMBDA_WARMUP_FRAC,
        )

    stats, global_step = run_epoch(
        train_loader,
        optimizer,
        scheduler,
        global_step,
        total_steps_p2,
        train_mode=True,
        grl_in_phase=True,
        lambda_fn=_p2_lambda,
        domain_loss_alpha=DOMAIN_LOSS_ALPHA,
    )
    lam_now = _p2_lambda(min(global_step, max(total_steps_p2 - 1, 0)))
    print(
        f"  P2 epoch {epoch+1}: loss={stats['loss']:.4f} λ≈{lam_now:.3f} (max {GRL_LAMBDA_MAX}) "
        f"diff={stats['difficulty']:.4f} dom={stats['domain']:.4f}"
    )
    m = evaluate_difficulty_macro_f1_and_domain_accuracy(model, val_loader, device)
    vf1 = m["macro_f1_difficulty"]
    print(f"    val macro-F1={vf1:.4f} domain_acc={m['domain_accuracy']:.4f}")
    if vf1 > best_val_f1:
        best_val_f1 = vf1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        stall = 0
        print(f"    -> new best val F1 {best_val_f1:.4f}")
    else:
        stall += 1
        if PHASE2_EARLY_STOP_PATIENCE > 0 and stall >= PHASE2_EARLY_STOP_PATIENCE:
            print("    early stop Phase 2")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best Phase-2 weights (val F1={best_val_f1:.4f})")


In [ ]:
results = {}
for split_name, loader in [("test", test_loader), ("val", val_loader), *ood_loaders.items()]:
    m = evaluate_difficulty_macro_f1_and_domain_accuracy(model, loader, device)
    results[split_name] = m
    print(
        f"{split_name:20s}  macro-F1={m['macro_f1_difficulty']:.4f}  "
        f"domain_acc={m['domain_accuracy']:.4f}"
    )

OUT_DIR = (
    DRIVE_SAVE_DIR
    if (RUNTIME == "colab" and SAVE_TO_DRIVE)
    else WORK_DIR / "electra_scalar_dann"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
torch.save(model.state_dict(), OUT_DIR / "model_weights.pt")
tokenizer.save_pretrained(OUT_DIR)
with open(OUT_DIR / "domain2id.json", "w") as f:
    json.dump(domain2id, f, indent=2)
train_meta = {
    "merge_coqa": MERGE_COQA_INTO_TRAIN,
    "n_train": len(df_train),
    "grl_lambda_max": GRL_LAMBDA_MAX,
    "domain_loss_alpha": DOMAIN_LOSS_ALPHA,
}
with open(OUT_DIR / "train_meta.json", "w") as f:
    json.dump(train_meta, f, indent=2)
print(f"\nSaved to {OUT_DIR}")
